# 데이터셋 코드 테스트 — `pm_safeline`

PM 세이프라인 데이터셋 파이프라인(사고 → OSM 지점 → negative → 로드뷰 이미지 → split)을
단계별로 검증하고 **데이터 개수**를 집계한다.

- 사고 데이터셋: `datasets.accidents` (KoROAD/TAAS)
- 데이터 primitive: `datasets.primitives` (geo/negatives/streetview)
- 로드뷰 데이터셋: `datasets.roadview` (수집 + torchvision Dataset + split)

> 수집 단계는 **torch 불필요**. 이미지 수집은 `mock` provider 로 빠르게 검증한다.

In [1]:
import os, sys
# src/main/python 을 import 경로에 추가(노트북이 notebooks/ 에 있으므로)
sys.path.insert(0, os.path.abspath(".."))
os.environ["PM_SV_PROVIDER"] = "mock"   # 이미지 수집: 네트워크/키 없이 검증

import pandas as pd
from pm_safeline.datasets.primitives.config import Config
cfg = Config()
print("data_dir =", cfg.data_dir)
print("bbox     =", cfg.bbox)

data_dir = C:\Users\BREW\Downloads\AI기반사회문제해결프로젝트\proj\data
bbox     = (127.3, 36.31, 127.43, 36.39)


## 1. 사고 데이터셋 — `load_accidents` / `AccidentDataset`

KoROAD 이륜차 다발지역을 표준 스키마 GeoDataFrame 으로 로드(캐시 사용).
`download=True` 로 하면 오픈API 에서 새로 받는다(인증키 필요).

In [2]:
from pm_safeline.datasets.accidents import load_accidents, AccidentDataset, CORE_COLUMNS, SEVERITY_LEVELS

acc = load_accidents("koroad", cfg, download=False)   # data/raw 캐시 사용
n_pos = len(acc)
print(f"사고 지점(positive): {n_pos}")
print("severity 분포:", acc["severity"].value_counts().to_dict())
print("표준 컬럼:", CORE_COLUMNS)
acc[["accident_id","lat","lon","severity","mode"]].head()

사고 지점(positive): 28
severity 분포: {'중상': 24, '경상': 4}
표준 컬럼: ['accident_id', 'datetime', 'lat', 'lon', 'severity', 'mode', 'geometry']


,accident_id,lat,lon,severity,mode
0,1,36.353050,127.368831,경상,motorcycle_frequentzone
1,2,36.351410,127.378420,경상,motorcycle_frequentzone
2,3,36.352113,127.376193,경상,motorcycle_frequentzone
3,4,36.348389,127.376663,경상,motorcycle_frequentzone
4,5,36.350989,127.425674,중상,motorcycle_frequentzone


In [3]:
# torchvision 표준 Dataset (AccidentDataset) 은 torch 필요 — 설치 시에만.
import importlib.util
if importlib.util.find_spec("torch"):
    ds = AccidentDataset(download=False, cfg=cfg)   # 캐시 accidents.gpkg 필요
    print("AccidentDataset len:", len(ds), "classes:", ds.classes)
else:
    print("torch 미설치 → AccidentDataset 인스턴스화 생략 (load_accidents 로 검증됨)")

torch 미설치 → AccidentDataset 인스턴스화 생략 (load_accidents 로 검증됨)


## 2. 데이터 primitive — `geo` (OSM 도로망 → 지점/스냅)

In [4]:
from pm_safeline.datasets.primitives import geo

edges = geo.load_drive_edges(cfg)
candidates = geo.sample_points_along_edges(edges, cfg)
acc_snapped = geo.snap_accidents_to_edges(acc, edges, cfg)
n_edges, n_cand = len(edges), len(candidates)
print(f"OSM edges: {n_edges}")
print(f"도로 후보 지점(고정간격 {cfg.sample_interval_m}m): {n_cand}")
print("사고 스냅 heading 예시:", acc_snapped[["heading","edge_id"]].head(3).to_dict("records"))

OSM edges: 22422
도로 후보 지점(고정간격 40.0m): 67875
사고 스냅 heading 예시: [{'heading': 129.5, 'edge_id': '8731'}, {'heading': 294.4, 'edge_id': '21767'}, {'heading': 114.9, 'edge_id': '19072'}]


## 3. 데이터 primitive — `negatives` (exposure-matched 대조지점)

In [5]:
from pm_safeline.datasets.primitives import negatives

negs = negatives.sample_negatives(acc_snapped, candidates, cfg)
labeled = negatives.build_labeled_points(acc_snapped, negs)
n_neg = int((labeled.label==0).sum()); n_pos2 = int((labeled.label==1).sum())
print(f"대조 지점(negative): {n_neg}   (negative_ratio={cfg.negative_ratio})")
print(f"총 라벨 지점: {len(labeled)}  (pos {n_pos2} / neg {n_neg})")
labeled[["point_id","label","lat","lon","heading"]].head()

대조 지점(negative): 84   (negative_ratio=3.0)
총 라벨 지점: 112  (pos 28 / neg 84)


,point_id,label,lat,lon,heading
0,acc_000000,1,36.353050,127.368831,129.5
1,acc_000001,1,36.351410,127.378420,294.4
2,acc_000002,1,36.352113,127.376193,114.9
3,acc_000003,1,36.348389,127.376663,31.6
4,acc_000004,1,36.350989,127.425674,133.3


## 4. 로드뷰 데이터셋 — 이미지 수집 (`roadview.collect_images`)

`mock` provider 로 지점당 이미지를 만들어 `manifest.csv` 를 생성한다.
(실제 수집은 `PM_SV_PROVIDER=naver/google`. 여기선 소량만.)

In [6]:
from pm_safeline.datasets import roadview

# 빠른 검증: 앞쪽 지점 일부만 mock 이미지 수집
subset = labeled.head(12).copy()
manifest = roadview.collect_images(subset, cfg, limit=12)
print(f"수집 이미지: {len(manifest)}  (지점 {subset['point_id'].nunique()}개)")
print("manifest 컬럼:", list(manifest.columns))
manifest.head()

[collect] 12/12 지점 처리
[collect] manifest 저장: C:\Users\BREW\Downloads\AI기반사회문제해결프로젝트\proj\data\manifest.csv (12 이미지)
수집 이미지: 12  (지점 12개)
manifest 컬럼: ['point_id', 'label', 'class', 'lat', 'lon', 'heading', 'severity', 'mode', 'path']


,point_id,label,class,lat,lon,heading,severity,mode,path
0,acc_000000,1,accident,36.353050,127.368831,129.5,경상,motorcycle_frequentzone,streetview\accident\acc_000000_h130.jpg
1,acc_000001,1,accident,36.351410,127.378420,294.4,경상,motorcycle_frequentzone,streetview\accident\acc_000001_h294.jpg
2,acc_000002,1,accident,36.352113,127.376193,114.9,경상,motorcycle_frequentzone,streetview\accident\acc_000002_h115.jpg
3,acc_000003,1,accident,36.348389,127.376663,31.6,경상,motorcycle_frequentzone,streetview\accident\acc_000003_h032.jpg
4,acc_000004,1,accident,36.350989,127.425674,133.3,중상,motorcycle_frequentzone,streetview\accident\acc_000004_h133.jpg


In [7]:
# 전체 수집 시 총 이미지 개수(예상): 지점당 heading 수 합 (heading 있으면 1, 없으면 4)
n_img_full = int(sum(1 if pd.notna(h) else 4 for h in labeled["heading"]))
print(f"전체 라벨 지점 {len(labeled)} → 수집 시 총 이미지(예상): {n_img_full}")

전체 라벨 지점 112 → 수집 시 총 이미지(예상): 112


## 5. 학습 분할 — `split_indices` / `kfold_indices`

지점(`point_id`) 단위 group 유지 + 라벨 stratify. **torch 불필요**(sklearn).

In [8]:
from pm_safeline.datasets.roadview import split_indices, kfold_indices

tr, va = split_indices(labeled, valid_frac=0.2, seed=42)
tp = set(labeled.iloc[tr].point_id); vp = set(labeled.iloc[va].point_id)
print(f"train {len(tr)}지점 / valid {len(va)}지점")
print("지점 누수(교집합):", tp & vp, "→ 없어야 정상")
print("valid 사고비율:", round(labeled.iloc[va].label.mean(),2), "/ train:", round(labeled.iloc[tr].label.mean(),2))
folds = kfold_indices(labeled, n_splits=5, seed=1)
leak = any(set(labeled.iloc[a].point_id) & set(labeled.iloc[b].point_id) for a,b in folds)
print(f"k-fold(5) 지점 누수: {leak} → False 여야 정상")

train 90지점 / valid 22지점
지점 누수(교집합): set() → 없어야 정상
valid 사고비율: 0.23 / train: 0.26
k-fold(5) 지점 누수: False → False 여야 정상


## 6. 요약 — 데이터 개수

In [9]:
summary = pd.DataFrame([
    ("사고 지점 (positive)", n_pos),
    ("대조 지점 (negative)", n_neg),
    ("총 라벨 지점", len(labeled)),
    ("총 로드뷰 이미지 (수집 시)", n_img_full),
    ("OSM edges", n_edges),
    ("도로 후보 지점", n_cand),
], columns=["항목", "개수"])
print(summary.to_string(index=False))

              항목    개수
사고 지점 (positive)    28
대조 지점 (negative)    84
         총 라벨 지점   112
총 로드뷰 이미지 (수집 시)   112
       OSM edges 22422
        도로 후보 지점 67875
